# Практична робота 1

## Google Colab, GitHub і перше знайомство з даними

**Дисципліна:** Розпізнавання образів і машинне навчання  
**Студент:** Река Євгеній  
**Група:** ІН-42/2  
**Дата виконання:** 16.11.2026

### Мета роботи

Ознайомитися із середовищем Google Colab, структурою ML-проєкту, сервісом GitHub і базовими операціями аналізу табличних даних.

## 1. Підключення бібліотек

На цьому етапі підключаються бібліотеки для роботи з таблицями, числовими масивами, графіками та наборами даних машинного навчання.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

from sklearn.datasets import load_iris
from IPython.display import display

Перевірка версій встановлених бібліотек. Версії в різні дні можуть відрізнятися — це не є помилкою.

In [ ]:
print("Версія Python:", sys.version.split()[0])
print("Версія NumPy:", np.__version__)
print("Версія pandas:", pd.__version__)
print("Версія scikit-learn:", sklearn.__version__)

## 2. Підключення Google Drive

Google Drive використовується для постійного зберігання наборів даних і результатів роботи. Файли у тимчасовій пам'яті Colab можуть бути видалені після завершення сеансу.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

### 2.1. Створення робочих папок на Google Drive

`mkdir(parents=True, exist_ok=True)` створює папку разом із батьківськими папками і не видає помилку, якщо папка вже існує.

In [ ]:
PROJECT_DIR = Path("/content/drive/MyDrive/pattern-recognition-practice")
DATA_DIR = PROJECT_DIR / "data"
NOTEBOOKS_DIR = PROJECT_DIR / "notebooks"
REPORTS_DIR = PROJECT_DIR / "reports"

DATA_DIR.mkdir(parents=True, exist_ok=True)
NOTEBOOKS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("Робоча папка:", PROJECT_DIR)
print("Папка даних:", DATA_DIR)
print("Папка Notebook:", NOTEBOOKS_DIR)
print("Папка звітів:", REPORTS_DIR)

Перевірка створених папок:

In [ ]:
for path in PROJECT_DIR.iterdir():
    print(path.name)

## 3. Завантаження набору даних

У роботі використовується навчальний набір Iris із бібліотеки scikit-learn. Він містить вимірювання квіток трьох видів ірису. Параметр `as_frame=True` повертає дані у вигляді об'єктів pandas.

In [ ]:
iris = load_iris(as_frame=True)

print(type(iris))
print(iris.keys())

Назви ознак і класів:

In [ ]:
print("Назви ознак:")
for feature_name in iris.feature_names:
    print("-", feature_name)

print("\nНазви класів:")
for target_name in iris.target_names:
    print("-", target_name)

Створення DataFrame. Метод `copy()` створює окрему копію таблиці, щоб зміни не впливали на початковий об'єкт.

In [ ]:
df = iris.frame.copy()

display(df.head())

### 3.1. Перетворення числового класу на назву виду

У початковому наборі класи закодовані числами 0, 1, 2. Замінимо їх на назви видів.

In [ ]:
print(df["target"].value_counts().sort_index())

In [ ]:
target_mapping = {
    0: "setosa",
    1: "versicolor",
    2: "virginica"
}

print(target_mapping)

# Нова текстова цільова змінна замість числового стовпця target
df["species"] = df["target"].map(target_mapping)
df = df.drop(columns=["target"])

### 3.2. Перейменування стовпців

In [ ]:
df = df.rename(
    columns={
        "sepal length (cm)": "sepal_length_cm",
        "sepal width (cm)": "sepal_width_cm",
        "petal length (cm)": "petal_length_cm",
        "petal width (cm)": "petal_width_cm"
    }
)

display(df.head())

## 4. Збереження даних у форматі CSV

Підготовлена таблиця зберігається на Google Drive, після чого повторно завантажується засобами бібліотеки pandas. Параметр `index=False` забороняє записувати до файла службовий індекс pandas.

In [ ]:
CSV_PATH = DATA_DIR / "iris.csv"

df.to_csv(CSV_PATH, index=False)

print("Файл збережено:")
print(CSV_PATH)

In [ ]:
if CSV_PATH.exists():
    print("Файл успішно створено.")
    print("Розмір файла:", CSV_PATH.stat().st_size, "байт")
else:
    print("Файл не знайдено.")

## 5. Читання табличних даних

Для читання CSV-файла використовується функція `pandas.read_csv()`. Результатом є об'єкт типу DataFrame.

In [ ]:
data = pd.read_csv(CSV_PATH)

print("Тип об'єкта:", type(data))

### 5.1. Перші та останні рядки

`head()` показує перші п'ять рядків, `tail()` — останні п'ять.

In [ ]:
display(data.head())

In [ ]:
display(data.tail())

### 5.2. Розмір таблиці та назви стовпців

Перший елемент `shape` — кількість рядків, другий — кількість стовпців.

In [ ]:
print("Розмір таблиці:", data.shape)
print("Кількість об'єктів:", data.shape[0])
print("Кількість стовпців:", data.shape[1])

In [ ]:
print("Назви стовпців:")

for column in data.columns:
    print("-", column)

### 5.3. Типи даних і загальна інформація

In [ ]:
print(data.dtypes)

In [ ]:
data.info()

### 5.4. Пропущені значення, дублікати та унікальні значення

In [ ]:
missing_values = data.isna().sum()

print("Кількість пропущених значень:")
print(missing_values)
print("Загальна кількість пропусків:", data.isna().sum().sum())

In [ ]:
duplicate_count = data.duplicated().sum()

print("Кількість повністю однакових рядків:", duplicate_count)

In [ ]:
print("Кількість унікальних значень у кожному стовпці:")
print(data.nunique())

### 5.5. Зведена таблиця структури даних

In [ ]:
structure_summary = pd.DataFrame(
    {
        "data_type": data.dtypes.astype(str),
        "missing_values": data.isna().sum(),
        "unique_values": data.nunique()
    }
)

display(structure_summary)

**Пояснення до таблиці структури:**

- Числові стовпці (тип `float64`): `sepal_length_cm`, `sepal_width_cm`, `petal_length_cm`, `petal_width_cm`.
- Категоріальний стовпець (текстовий тип): `species` — вид ірису.
- Пропущених значень немає у жодному стовпці.
- Найменше унікальних значень має `species` — лише 3 (це класи). Серед числових ознак найменше унікальних значень у `petal_width_cm` (22) і `sepal_width_cm` (23), бо вимірювання записані з точністю до 0,1 см.
- Знайдено 1 повністю однаковий рядок (дві квітки virginica з однаковими вимірами). Його не видаляємо, лише фіксуємо.

## 6. Описова статистика

`describe()` для числових стовпців показує кількість значень, середнє, стандартне відхилення, мінімум, квартилі (25 %, 50 % — медіана, 75 %) і максимум. `.T` транспонує таблицю для зручного перегляду.

In [ ]:
numeric_statistics = data.describe().T

display(numeric_statistics)

### 6.1. Статистика цільової змінної

In [ ]:
class_counts = data["species"].value_counts()

display(class_counts.rename("count").to_frame())

In [ ]:
class_percentages = (
    data["species"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

display(class_percentages.rename("percentage").to_frame())

## 7. Візуалізація даних

Гістограми використовуються для первинного дослідження розподілів числових ознак.

In [ ]:
feature_columns = [
    "sepal_length_cm",
    "sepal_width_cm",
    "petal_length_cm",
    "petal_width_cm"
]

data[feature_columns].hist(
    bins=15,
    figsize=(10, 8)
)

plt.suptitle("Розподіли числових ознак набору Iris")
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

**Аналіз гістограм:**

1. **Найменші значення** має ознака `petal_width_cm` (ширина пелюстки): від 0,1 до 2,5 см.
2. **Найбільший діапазон** має `petal_length_cm` (довжина пелюстки): від 1,0 до 6,9 см, тобто 5,9 см.
3. **Форма розподілів різна.** Ознаки чашолистка (`sepal_length_cm`, `sepal_width_cm`) мають один пік і схожі на нормальний розподіл. Ознаки пелюстки розподілені нерівномірно і мають розрив.
4. **Кілька груп значень** видно у `petal_length_cm` і `petal_width_cm`: окрема група малих значень (довжина приблизно 1–2 см, ширина 0,1–0,6 см) і друга група більших значень. Найімовірніше, мала група — це один вид ірису.

### 7.1. Розподіл об'єктів за класами

In [ ]:
class_counts = data["species"].value_counts().sort_index()

class_counts.plot(
    kind="bar",
    figsize=(8, 5)
)

plt.title("Кількість об'єктів кожного класу")
plt.xlabel("Вид ірису")
plt.ylabel("Кількість об'єктів")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.3)
plt.show()

**Чи є набір даних збалансованим?** Так. Кожен із трьох класів містить рівно 50 об'єктів (по 33,33 %), тому набір повністю збалансований.

### 7.2. Діаграма залежності довжини та ширини пелюстки

In [ ]:
# Кожен вид малюємо окремим кольором, щоб побачити групи
for species_name, group in data.groupby("species"):
    plt.scatter(
        group["petal_length_cm"],
        group["petal_width_cm"],
        label=species_name,
        alpha=0.8
    )

plt.title("Залежність довжини та ширини пелюстки")
plt.xlabel("Довжина пелюстки, см")
plt.ylabel("Ширина пелюстки, см")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**Аналіз діаграми:**

1. **Так, об'єкти різних класів утворюють окремі групи.** Точки кожного виду зосереджені у своїй області.
2. **Найкраще відокремлюється setosa.** Її пелюстки короткі (до 2 см) і вузькі (до 0,6 см), а група лежить далеко від інших.
3. **Частково перекриваються versicolor і virginica**, приблизно в області довжини пелюстки 4,5–5,1 см і ширини 1,5–1,8 см.
4. **Так, ці ознаки можна використати для розпізнавання виду.** Setosa визначається майже безпомилково, а versicolor і virginica розрізняються в більшості випадків, крім зони перекриття.

## 8. Постановка задачі машинного навчання

### Об'єкт дослідження

Одним об'єктом є одна квітка ірису, для якої виміряно характеристики чашолистка та пелюстки.

### Ознаки

Використовуються чотири числові ознаки:

1. довжина чашолистка;
2. ширина чашолистка;
3. довжина пелюстки;
4. ширина пелюстки.

### Цільова змінна

Цільовою змінною є `species` – вид квітки ірису.

### Тип задачі

Задача належить до навчання з учителем і є задачею багатокласової класифікації.

### Формулювання задачі

Необхідно за чотирма числовими характеристиками квітки автоматично визначити, до якого з трьох видів ірису вона належить: setosa, versicolor або virginica.

### 8.1. Програмне розділення ознак і цільової змінної

На цьому етапі моделі ще не навчаються. `X` — матриця ознак, `y` — цільова змінна.

In [ ]:
X = data.drop(columns=["species"])
y = data["species"]

print("Розмір матриці ознак X:", X.shape)
print("Розмір цільової змінної y:", y.shape)

In [ ]:
display(X.head())

In [ ]:
display(y.head())

## 9. Автоматична перевірка правильності результату

Якщо будь-яка умова не виконується, `assert` зупиняє виконання з помилкою `AssertionError`.

In [ ]:
expected_columns = {
    "sepal_length_cm",
    "sepal_width_cm",
    "petal_length_cm",
    "petal_width_cm",
    "species"
}

assert data.shape == (150, 5), (
    f"Очікувався розмір (150, 5), отримано {data.shape}"
)

assert set(data.columns) == expected_columns, (
    "Назви стовпців не відповідають очікуваним."
)

assert data.isna().sum().sum() == 0, (
    "У таблиці знайдено пропущені значення."
)

assert data["species"].nunique() == 3, (
    "Очікувалося три класи."
)

assert set(data["species"].unique()) == {
    "setosa",
    "versicolor",
    "virginica"
}, "Назви класів не відповідають очікуваним."

print("Усі основні перевірки виконано успішно.")

## 10. Додаткове завдання (варіант 1)

Діаграма залежності довжини чашолистка (`sepal_length_cm`) від його ширини (`sepal_width_cm`) для трьох видів ірису.

In [ ]:
for species_name, group in data.groupby("species"):
    plt.scatter(
        group["sepal_width_cm"],
        group["sepal_length_cm"],
        label=species_name,
        alpha=0.8
    )

plt.title("Залежність довжини чашолистка від його ширини")
plt.xlabel("Ширина чашолистка, см")
plt.ylabel("Довжина чашолистка, см")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**Висновок до діаграми:** setosa і тут утворює окрему групу: у неї короткі, але широкі чашолистки. Versicolor і virginica за ознаками чашолистка сильно перекриваються, тому для їх розрізнення ознаки пелюстки підходять значно краще.

# Висновки

Під час виконання практичної роботи було створено GitHub-репозиторій для зберігання матеріалів курсу та сформовано базову структуру ML-проєкту.

У середовищі Google Colab було підключено бібліотеки NumPy, pandas, Matplotlib і scikit-learn, а також підключено Google Drive. Для дослідження використано набір даних Iris, який містить 150 об'єктів, чотири числові ознаки та одну цільову категоріальну змінну. Об'єктом дослідження є квітка ірису, а цільовою змінною – її вид. Під час первинного аналізу було визначено розмір таблиці, назви стовпців, типи даних, кількість пропущених значень і основні статистичні характеристики. Також було побудовано гістограми ознак, діаграму кількості об'єктів кожного класу та діаграму залежності довжини й ширини пелюстки.

Встановлено, що досліджувана задача є задачею багатокласової класифікації: за чотирма числовими ознаками необхідно визначити один із трьох видів ірису.

**Власні спостереження:**

- Пропущених значень у наборі немає. Виявлено один повністю однаковий рядок, який на цьому етапі не видалявся.
- Класи повністю збалансовані: по 50 об'єктів (33,33 %) у кожному.
- Найкраще розділяють класи ознаки пелюстки — `petal_length_cm` і `petal_width_cm`. На їхніх гістограмах видно окремі групи значень, а на діаграмі розсіювання кожен вид займає свою область.
- Setosa відокремлюється від інших видів повністю. Versicolor і virginica частково перекриваються на діаграмі пелюсток, а за ознаками чашолистка перекриваються ще сильніше.

# Відповіді на контрольні запитання

1. **Що таке Google Colab?** Хмарне середовище від Google, у якому можна створювати й запускати Jupyter Notebook у браузері без встановлення Python на комп'ютер.

2. **Що таке Jupyter Notebook?** Інтерактивний документ (файл `.ipynb`), що поєднує комірки з кодом, результати їх виконання (таблиці, графіки) і текстові пояснення.

3. **Які два основні типи комірок використовуються в Notebook?** Комірки з кодом (Code) і текстові комірки (Text / Markdown).

4. **Як запустити комірку?** Кнопкою запуску ліворуч від комірки або комбінацією клавіш Shift + Enter.

5. **Для чого призначений файл README.md?** Це основний опис репозиторію: назва, автор, призначення, структура, опис робіт, інструкція із запуску. Його першим бачить користувач на GitHub.

6. **Що таке pandas DataFrame?** Двовимірна таблиця бібліотеки pandas з іменованими стовпцями та індексом рядків. Різні стовпці можуть мати різні типи даних.

7. **Що є рядком таблиці?** Один об'єкт (приклад) — у наборі Iris це одна квітка.

8. **Що є стовпцем таблиці?** Одна характеристика (ознака) всіх об'єктів або цільова змінна.

9. **Що таке об'єкт у задачі машинного навчання?** Один досліджуваний приклад, який описується набором ознак і відповідає одному рядку таблиці.

10. **Що таке ознака?** Вимірювана характеристика об'єкта, якій відповідає окремий стовпець таблиці.

11. **Що таке цільова змінна?** Значення, яке модель має навчитися прогнозувати за ознаками.

12. **Що є об'єктом у наборі Iris?** Одна квітка ірису.

13. **Які ознаки містить набір Iris?** Довжина й ширина чашолистка, довжина й ширина пелюстки (усі в сантиметрах).

14. **Яка цільова змінна використовується у роботі?** `species` — вид ірису.

15. **Чому задача Iris є задачею класифікації?** Цільова змінна є категорією (вид квітки), а не числом, тож потрібно віднести об'єкт до одного зі скінченного набору класів.

16. **Чому ця задача є багатокласовою?** Класів більше двох — три: setosa, versicolor і virginica.

17. **Для чого використовується метод head()?** Для виведення перших рядків таблиці (за замовчуванням п'яти).

18. **Для чого використовується метод tail()?** Для виведення останніх рядків таблиці (за замовчуванням п'яти).

19. **Що повертає властивість shape?** Кортеж (кількість рядків, кількість стовпців). Для Iris це (150, 5).

20. **Для чого використовується метод info()?** Показує кількість рядків, назви стовпців, кількість непорожніх значень, типи даних і обсяг пам'яті.

21. **Яку інформацію повертає метод describe()?** Для числових стовпців: count, mean, std, min, 25 %, 50 % (медіану), 75 % і max.

22. **Як знайти пропущені значення?** За допомогою `data.isna().sum()` (по стовпцях) або `data.isna().sum().sum()` (загальна кількість).

23. **Як перевірити наявність дублікатів?** За допомогою `data.duplicated().sum()` — кількість рядків, які повністю повторюють попередні.

24. **Для чого використовуються гістограми?** Щоб побачити розподіл значень числової ознаки: діапазон, форму розподілу, піки, групи значень і викиди.

25. **Чому важливо аналізувати розподіл об'єктів за класами?** Щоб з'ясувати, чи збалансований набір. Якщо один клас переважає, модель може «звикнути» прогнозувати саме його, а метрика accuracy буде оманливою.

26. **Чим ознаки X відрізняються від цільової змінної y?** X — вхідні дані (матриця ознак, з якої робиться прогноз), y — те, що потрібно спрогнозувати (правильні відповіді).

27. **Чому важливі дані потрібно зберігати на Google Drive або GitHub?** Середовище Colab тимчасове: після завершення або перезапуску сеансу файли і змінні з нього зникають.

28. **Чому перед здаванням необхідно повторно виконати всі комірки?** Щоб переконатися, що Notebook працює зверху вниз без помилок, а всі результати й графіки збережені у файлі.

29. **Які файли містить створений навчальний ML-проєкт?** `README.md`, `.gitignore`, папку `notebooks/` (README.md і цей Notebook), `data/` (README.md і iris.csv) та `reports/` (README.md).